# Reasoning and Planning in CrewAI [Step 4 -- Chain-of-thought and task decomposition]

> **MLCourse - Agentic AI - CrewAI Advanced Agents**

By default, CrewAI agents produce answers in a single forward pass. Enabling
`reasoning` adds chain-of-thought steps where the agent thinks through the
problem before answering. Enabling `planning` adds a crew-level decomposition
step that generates a task execution plan before any task runs.

## What you will learn

- `agent.reasoning = True`: enable chain-of-thought on individual agents
- `max_reasoning_attempts`: cap how many reasoning steps an agent can take
- `crew.planning = True`: generate a task execution plan before running tasks
- How reasoning changes the agent's output (visible thinking traces)
- Comparing crew output with and without reasoning enabled
- When reasoning helps vs when it wastes tokens

In [ ]:
# === SETUP CELL ===
import os
import time
from pathlib import Path

from dotenv import load_dotenv

# Walk up from cwd until we reach the track root folder "03_agentic_ai".
# This lets the notebook run from any subfolder while finding the shared .env.
TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

# Guard Jupyter-only magic so this file stays valid as plain Python too.
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

print("Setup complete. Track root resolved to:", TRACK)

## 1. Agent reasoning -- chain-of-thought output

Setting `reasoning=True` on an agent enables chain-of-thought (CoT) reasoning.
Before producing a final answer, the agent outputs intermediate thinking steps
-- essentially "showing its work." This is useful for:

- **Complex analysis**: multi-step problems where the reasoning path matters
- **Debugging**: seeing WHY the agent chose a particular answer
- **Accuracy**: CoT reduces logical errors on arithmetic, comparison, and planning tasks

The `max_reasoning_attempts` parameter caps how many reasoning steps the agent
can take before being forced to produce a final answer. Default is 3; set it
higher for genuinely complex problems.

In [ ]:
from crewai import Agent, Task, Crew, Process

# Agent WITHOUT reasoning -- produces answers directly.
fast_agent = Agent(
    role="Quick Responder",
    goal="Answer questions directly and concisely.",
    backstory="You are fast and direct. No need to explain your thinking.",
    reasoning=False,  # Default -- single forward pass, no CoT.
    max_reasoning_attempts=0,
    llm="ollama/llama3.1:8b",
    verbose=False,
    allow_delegation=False,
)

# Agent WITH reasoning -- thinks through problems step by step.
reasoning_agent = Agent(
    role="Analytical Thinker",
    goal="Think through problems carefully before answering.",
    backstory=(
        "You are a methodical analyst. You always explain your reasoning "
        "step by step before reaching a conclusion."
    ),
    reasoning=True,  # Enable chain-of-thought.
    max_reasoning_attempts=3,  # Cap at 3 thinking steps.
    llm="ollama/llama3.1:8b",
    verbose=False,
    allow_delegation=False,
)

print("fast_agent.reasoning:", fast_agent.reasoning)
print("reasoning_agent.reasoning:", reasoning_agent.reasoning)
print("reasoning_agent.max_reasoning_attempts:", reasoning_agent.max_reasoning_attempts)

## 2. Side-by-side comparison -- same task, different reasoning

Run the same task through both agents to see how reasoning changes the output.
The reasoning agent will produce visible thinking steps before its answer,
while the fast agent jumps directly to the conclusion.

In [ ]:
analysis_task_fast = Task(
    description=(
        "A company has 150 employees. 40% are engineers, 25% are designers, "
        "and the rest are in operations. If the company hires 50 more engineers "
        "and lays off 10 operations staff, what percentage of the new total "
        "will be engineers?"
    ),
    expected_output="The final percentage, with clear reasoning shown.",
    agent=fast_agent,
)

analysis_task_reasoning = Task(
    description=(
        "A company has 150 employees. 40% are engineers, 25% are designers, "
        "and the rest are in operations. If the company hires 50 more engineers "
        "and lays off 10 operations staff, what percentage of the new total "
        "will be engineers?"
    ),
    expected_output="The final percentage, with clear reasoning shown.",
    agent=reasoning_agent,
)

# Run without reasoning first.
crew_fast = Crew(
    agents=[fast_agent],
    tasks=[analysis_task_fast],
    process=Process.sequential,
    verbose=False,
)

print("=" * 60)
print("WITHOUT REASONING (fast_agent)")
print("=" * 60)
start = time.time()
try:
    result_fast = crew_fast.kickoff()
    elapsed_fast = time.time() - start
    print(result_fast)
    print(f"\n[elapsed: {elapsed_fast:.1f}s]")
except Exception as e:
    print(f"[demo skipped] {e}")
    elapsed_fast = 0

Now run with reasoning enabled -- observe the chain-of-thought steps in the output.

In [ ]:
crew_reasoning = Crew(
    agents=[reasoning_agent],
    tasks=[analysis_task_reasoning],
    process=Process.sequential,
    verbose=False,
)

print("=" * 60)
print("WITH REASONING (reasoning_agent)")
print("=" * 60)
start = time.time()
try:
    result_reasoning = crew_reasoning.kickoff()
    elapsed_reasoning = time.time() - start
    print(result_reasoning)
    print(f"\n[elapsed: {elapsed_reasoning:.1f}s]")
except Exception as e:
    print(f"[demo skipped] {e}")
    elapsed_reasoning = 0

## 3. `max_reasoning_attempts` -- limiting thinking depth

Chain-of-thought can consume many tokens if the agent gets stuck in a loop.
`max_reasoning_attempts` sets a hard cap on reasoning iterations. After the
cap is reached, the agent produces its best answer from whatever reasoning
it has accumulated.

**Guidelines:**
- Simple questions: 0-1 attempts (reasoning adds latency without accuracy)
- Moderate analysis: 2-3 attempts (sweet spot for most tasks)
- Complex multi-step problems: 3-5 attempts (diminishing returns beyond 5)

In [ ]:
# Demonstrate different max_reasoning_attempts values.
for max_attempts in [1, 3, 5]:
    agent = Agent(
        role="Thinker",
        goal="Reason through problems carefully.",
        backstory="You are a careful thinker.",
        reasoning=True,
        max_reasoning_attempts=max_attempts,
        llm="ollama/llama3.1:8b",
        verbose=False,
    )

    task = Task(
        description=(
            "If a train travels at 60 mph for 2.5 hours, then at 80 mph for "
            "1.5 hours, what is the total distance traveled?"
        ),
        expected_output="The total distance with reasoning steps.",
        agent=agent,
    )

    crew = Crew(
        agents=[agent],
        tasks=[task],
        process=Process.sequential,
        verbose=False,
    )

    try:
        result = crew.kickoff()
        print(f"\n[max_reasoning_attempts={max_attempts}]")
        print(str(result)[:200] + "...")
    except Exception as e:
        print(f"[demo skipped] max_attempts={max_attempts}: {e}")

## 4. Crew-level planning -- task decomposition before execution

Setting `planning=True` on the Crew triggers a planning step BEFORE any task
runs. The planning LLM examines all tasks and generates a decomposition plan
describing the execution strategy. This plan is shown to the user and used
to guide task execution.

**When planning helps:**
- Tasks with dependencies (task 2 needs task 1's output)
- Complex workflows where execution order is not obvious
- Multi-agent crews where task assignment matters

**When planning is overhead:**
- Simple sequential tasks with obvious order
- Single-agent crews with one task
- Time-sensitive pipelines where every millisecond counts

In [ ]:
planning_agent = Agent(
    role="Project Manager",
    goal="Break down complex projects into clear, executable steps.",
    backstory=(
        "You are an experienced project manager who creates detailed "
        "execution plans before any work begins."
    ),
    llm="ollama/llama3.1:8b",
    verbose=False,
)

# Define a multi-step project task.
research_task = Task(
    description=(
        "Research the current market size for AI-powered customer support "
        "tools. Provide specific dollar figures and growth rates."
    ),
    expected_output="Market size data with specific numbers and sources.",
    agent=planning_agent,
)

analysis_task = Task(
    description=(
        "Based on the market research, analyze the top 3 competitors and "
        "their key differentiators. Create a comparison matrix."
    ),
    expected_output="A competitor comparison matrix with key differentiators.",
    agent=planning_agent,
)

strategy_task = Task(
    description=(
        "Using the market research and competitor analysis, propose a "
        "go-to-market strategy with 3 specific recommendations."
    ),
    expected_output="Three specific go-to-market recommendations with rationale.",
    agent=planning_agent,
)

# Crew with planning enabled.
planning_crew = Crew(
    agents=[planning_agent],
    tasks=[research_task, analysis_task, strategy_task],
    process=Process.sequential,
    planning=True,  # Enable the planning step.
    verbose=False,
)

print("Crew assembled with planning=True")
print("Before execution, CrewAI will generate a decomposition plan")
print("showing how the 3 tasks will be approached.\n")

try:
    result = planning_crew.kickoff()
    print("=== Planning Crew Result ===")
    print(result)
except Exception as e:
    print(f"[demo skipped] {e}")

## 5. Planning WITHOUT the planning step -- for comparison

Run the same tasks without planning to see the difference. Without planning,
CrewAI runs tasks sequentially in the order you defined them, with no
upfront decomposition or strategy. The agent just starts working on task 1,
then task 2, etc.

In [ ]:
no_planning_crew = Crew(
    agents=[planning_agent],
    tasks=[research_task, analysis_task, strategy_task],
    process=Process.sequential,
    planning=False,  # No planning step -- straight to execution.
    verbose=False,
)

print("Same tasks, planning=False -- no decomposition step")
print("Agent starts executing immediately without a strategy overview.\n")

try:
    result = no_planning_crew.kickoff()
    print("=== No-Planning Crew Result ===")
    print(result)
except Exception as e:
    print(f"[demo skipped] {e}")

## 6. Combining reasoning AND planning

For maximum accuracy on complex multi-step tasks, combine both features:
`crew.planning=True` generates the execution strategy, and
`agent.reasoning=True` ensures each agent thinks through its individual task.

This is the highest-quality (but slowest) configuration. Use it when accuracy
matters more than speed.

In [ ]:
# Agent with both reasoning enabled.
deep_agent = Agent(
    role="Deep Analyst",
    goal="Analyze problems thoroughly with structured reasoning.",
    backstory=(
        "You combine careful planning with deep individual reasoning. "
        "You never rush to conclusions."
    ),
    reasoning=True,
    max_reasoning_attempts=3,
    llm="ollama/llama3.1:8b",
    verbose=False,
)

deep_task = Task(
    description=(
        "Evaluate the pros and cons of migrating from a monolithic architecture "
        "to microservices for a startup with 5 engineers and 10,000 daily users. "
        "Consider: development speed, operational complexity, team expertise, "
        "and long-term scalability. Provide a balanced recommendation."
    ),
    expected_output=(
        "A balanced evaluation with pros, cons, and a clear recommendation "
        "with reasoning for each point."
    ),
    agent=deep_agent,
)

deep_crew = Crew(
    agents=[deep_agent],
    tasks=[deep_task],
    process=Process.sequential,
    planning=True,  # Crew-level planning.
    verbose=False,
)

print("Running with BOTH reasoning AND planning enabled")
print("(This is the highest quality but slowest configuration)\n")

start = time.time()
try:
    result = deep_crew.kickoff()
    elapsed = time.time() - start
    print("=== Deep Analysis Result ===")
    print(result)
    print(f"\n[elapsed: {elapsed:.1f}s]")
except Exception as e:
    print(f"[demo skipped] {e}")

## 7. When to use reasoning vs planning

| Feature | What it does | Cost | Best for |
|---|---|---|---|
| `reasoning=True` | Agent thinks step-by-step before answering | +50-100% tokens per task | Complex individual analysis |
| `planning=True` | Crew generates execution plan before starting | One extra LLM call upfront | Multi-task workflows with dependencies |
| Both | Full quality pipeline | 2-3x total cost | Critical decisions, research reports |
| Neither | Fast, direct execution | Baseline cost | Simple tasks, prototypes, speed-critical |

**Rule of thumb:** start without either feature. Add reasoning when agents
produce wrong answers on analytically demanding tasks. Add planning when
task ordering or dependencies cause issues.

In [ ]:
# Quick benchmark: measure token/time overhead of each configuration.
configs = [
    ("No reasoning, no planning", False, False),
    ("Reasoning only", True, False),
    ("Planning only", False, True),
    ("Both reasoning and planning", True, True),
]

for label, use_reasoning, use_planning in configs:
    agent = Agent(
        role="Tester",
        goal="Answer a simple question.",
        backstory="You answer directly.",
        reasoning=use_reasoning,
        max_reasoning_attempts=2,
        llm="ollama/llama3.1:8b",
        verbose=False,
    )

    task = Task(
        description="What is 12 multiplied by 15?",
        expected_output="The answer with brief reasoning if enabled.",
        agent=agent,
    )

    crew = Crew(
        agents=[agent],
        tasks=[task],
        process=Process.sequential,
        planning=use_planning,
        verbose=False,
    )

    start = time.time()
    try:
        result = crew.kickoff()
        elapsed = time.time() - start
        print(f"[{label}] {elapsed:.1f}s -- {str(result)[:80]}...")
    except Exception as e:
        print(f"[{label}] skipped: {e}")

## Summary and key takeaways

- `agent.reasoning = True` enables chain-of-thought: the agent shows its
  thinking steps before producing the final answer.
- `max_reasoning_attempts` caps reasoning depth (default 3); higher values
  for complex problems, lower for simple ones.
- `crew.planning = True` adds a crew-level decomposition step that generates
  an execution strategy before any task runs.
- Reasoning improves accuracy on analytical tasks; planning improves coherence
  on multi-task workflows.
- Combining both gives maximum quality but 2-3x the token cost and latency.
- Start simple: add reasoning or planning only when you see specific quality
  issues that warrant the overhead.

**Next up:** notebook 05 covers Conditional Tasks and Multimodal Agents.